# Train Autocomplete v2 (SentencePiece + GRU)

This notebook retrains **SentencePiece v2** and **GRU v2** with:
- SentencePiece BPE (vocab = 24k)
- Context length = 8
- GRU language model


In [1]:
import os
import numpy as np
import tensorflow as tf
import sentencepiece as spm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense


In [2]:
# CONFIG (v2 FINAL)
DATA_PATH = "../data.txt"

SP_MODEL_PREFIX = "sentencepiece_autocomplete_v2"
SP_MODEL_PATH = "sentencepiece_autocomplete_v2.model"
MODEL_PATH = "gru_autocomplete_sentencepiece_v2.h5"

SEQ_LEN = 8
BATCH_SIZE = 1024
BUFFER_SIZE = 10000
EMBED_DIM = 128
GRU_UNITS = 256

EPOCHS = 3
STEPS_PER_EPOCH = 15000


In [3]:
# Train SentencePiece v2 (run once)
if not os.path.exists(SP_MODEL_PATH):
    print("Training SentencePiece v2...")
    spm.SentencePieceTrainer.train(
        input=DATA_PATH,
        model_prefix=SP_MODEL_PREFIX,
        vocab_size=24000,
        model_type="bpe",
        character_coverage=0.9995,
        input_sentence_size=5_000_000,
        shuffle_input_sentence=True
    )
    print("SentencePiece training completed.")
else:
    print("SentencePiece model already exists. Skipping training.")


SentencePiece model already exists. Skipping training.


In [4]:
# Load SentencePiece tokenizer
sp = spm.SentencePieceProcessor()
sp.load(SP_MODEL_PATH)

VOCAB_SIZE = sp.get_piece_size()
print("SentencePiece vocab size:", VOCAB_SIZE)


SentencePiece vocab size: 24000


In [5]:
# Dataset generator
def sp_generator():
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            ids = sp.encode(line.strip(), out_type=int)
            if len(ids) < SEQ_LEN + 1:
                continue
            for i in range(SEQ_LEN, len(ids)):
                yield ids[i-SEQ_LEN:i], ids[i]


In [6]:
dataset = tf.data.Dataset.from_generator(
    sp_generator,
    output_signature=(
        tf.TensorSpec(shape=(SEQ_LEN,), dtype=tf.int32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )
)

dataset = (
    dataset.shuffle(BUFFER_SIZE)
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE)
)


In [7]:
# Build GRU model
model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=SEQ_LEN),
    GRU(GRU_UNITS),
    Dense(256, activation="relu"),
    Dense(VOCAB_SIZE, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 8, 128)            3072000   
                                                                 
 gru (GRU)                   (None, 256)               296448    
                                                                 
 dense (Dense)               (None, 256)               65792     
                                                                 
 dense_1 (Dense)             (None, 24000)             6168000   
                                                                 
Total params: 9,602,240
Trainable params: 9,602,240
Non-trainable params: 0
_________________________________________________________________


In [8]:
# Train GRU v2
model.fit(
    dataset,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    verbose=1
)


Epoch 1/3
15000/15000 [==============================] - 3190s 212ms/step - loss: 5.2343 - accuracy: 0.2271
Epoch 2/3
15000/15000 [==============================] - 2885s 192ms/step - loss: 4.4300 - accuracy: 0.2985
Epoch 3/3
15000/15000 [==============================] - 2893s 193ms/step - loss: 4.3756 - accuracy: 0.2931


In [9]:
# Save trained model
model.save(MODEL_PATH)
print("Saved model to", MODEL_PATH)


Saved model to gru_autocomplete_sentencepiece_v2.h5
